In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_csv("qoute_dataset.csv")

In [4]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [5]:
df['quote'][0]

'“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

In [6]:
df.shape

(3038, 2)

In [7]:
quotes = df['quote']

In [8]:
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


In [9]:
import string
translator = str.maketrans('', '', string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))
quotes = df['quote'].apply(lambda x: x.translate(translator)).str.lower()

In [10]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


In [11]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [12]:
vocab_size = 8978
tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(quotes)

In [13]:
word_index = tokenizer.word_index
print(len(word_index))
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [14]:
sequence = tokenizer.texts_to_sequences(quotes)

In [15]:
quotes[0]

'“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”'

In [16]:
sequence[0]

[713,
 62,
 29,
 19,
 16,
 946,
 10,
 7,
 5,
 1156,
 8,
 70,
 293,
 10,
 145,
 12,
 809,
 104,
 752,
 70,
 2461]

In [17]:
X=[]
y=[]

for seq in sequence:
  for i in range(1,len(seq)):
    input_seq = seq[:i]
    output_seq = seq[i]
    X.append(input_seq)
    y.append(output_seq)

In [18]:
len(X)

85270

In [19]:
len(y)

85270

In [20]:
max_len = max(len(x) for x in X)
print(max_len)

745


In [21]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [22]:
X_padded = pad_sequences(X, maxlen=max_len, padding='pre')

In [23]:
y = np.array(y)

In [24]:
X_padded.shape

(85270, 745)

In [25]:
from tensorflow.keras.utils import to_categorical

In [26]:
y_one_hot = to_categorical(y,num_classes=vocab_size)

In [27]:
y.shape

(85270,)

In [28]:
y_one_hot.shape

(85270, 8978)

In [29]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,SimpleRNN

In [30]:
embedding_dim = 50
rnn_units = 128

In [31]:
rnn_model = Sequential()

rnn_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
rnn_model.add(SimpleRNN(units=rnn_units))
rnn_model.add(Dense(units=vocab_size, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [32]:

rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [33]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [34]:
lstm_model = Sequential()
lstm_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size, activation='softmax'))

In [35]:

lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [36]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [37]:
epochs = 100
batch_size = 128

rnn_history = rnn_model.fit(
   X_padded, y_one_hot,
  epochs=epochs,
  batch_size=batch_size,
  validation_split=0.1
)

Epoch 1/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 47s 71ms/step - accuracy: 0.0459 - loss: 6.6828 - val_accuracy: 0.0593 - val_loss: 6.4898
Epoch 2/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 40s 66ms/step - accuracy: 0.0827 - loss: 6.0490 - val_accuracy: 0.0938 - val_loss: 6.2857
Epoch 3/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 39s 65ms/step - accuracy: 0.1043 - loss: 5.7061 - val_accuracy: 0.1027 - val_loss: 6.2743
Epoch 4/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 39s 64ms/step - accuracy: 0.1202 - loss: 5.4241 - val_accuracy: 0.1075 - val_loss: 6.3060
Epoch 5/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 39s 65ms/step - accuracy: 0.1334 - loss: 5.1686 - val_accuracy: 0.1116 - val_loss: 6.3614
Epoch 6/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 39s 65ms/step - accuracy: 0.1447 - loss: 4.9334 - val_accuracy: 0.1111 - val_loss: 6.4081
Epoch 7/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 39s 64ms/step - accuracy: 0.1592 - loss: 4.7135 - val_accuracy: 0.1118 - val_loss: 6.4899
Epoch 8/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 65ms/step - accuracy: 0.1760 - loss: 4

In [38]:
epochs = 100
batch_size = 128

lstm_history = lstm_model.fit(
   X_padded, y_one_hot,
  epochs=epochs,
  batch_size=batch_size,
  validation_split=0.1
)

Epoch 1/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 39s 56ms/step - accuracy: 0.0386 - loss: 6.7422 - val_accuracy: 0.0453 - val_loss: 6.6629
Epoch 2/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 55ms/step - accuracy: 0.0600 - loss: 6.2981 - val_accuracy: 0.0698 - val_loss: 6.4973
Epoch 3/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 54ms/step - accuracy: 0.0833 - loss: 6.0180 - val_accuracy: 0.0864 - val_loss: 6.4255
Epoch 4/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 55ms/step - accuracy: 0.0985 - loss: 5.8056 - val_accuracy: 0.0958 - val_loss: 6.3896
Epoch 5/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 54ms/step - accuracy: 0.1110 - loss: 5.6168 - val_accuracy: 0.1038 - val_loss: 6.3625
Epoch 6/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 54ms/step - accuracy: 0.1225 - loss: 5.4352 - val_accuracy: 0.1068 - val_loss: 6.3769
Epoch 7/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 54ms/step - accuracy: 0.1312 - loss: 5.2691 - val_accuracy: 0.1113 - val_loss: 6.3921
Epoch 8/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 54ms/step - accuracy: 0.1404 - loss: 5

In [69]:
from tensorflow.keras.models import load_model

In [80]:
lstm_model = load_model("lstm_model.h5")

In [81]:
lstm_model.save("lstm_model.h5")

In [72]:
index_to_word = {}
for word, index in word_index.items():
  index_to_word[index] = word

In [73]:

from tensorflow.keras.preprocessing.sequence import pad_sequences

In [74]:
def predictor(model,tokenizer,text,max_len):
  text = text.lower()

  seq = tokenizer.texts_to_sequences([text])[0]
  seq = pad_sequences([seq], maxlen=max_len, padding='pre')

  pred = model.predict(seq,verbose = 0)
  pred_index = np.argmax(pred)
  return index_to_word[pred_index]

In [75]:
seed_text = "what are you"
next_word = predictor(lstm_model,tokenizer,seed_text,max_len)
print(next_word)

implying


In [76]:
def generate_text(model,tokenizer,seed_text,max_len,n_words):
  for _ in range(n_words):
    next_word = predictor(model,tokenizer,seed_text,max_len)
    if next_word == "":
      break
    seed_text += " " + next_word
  return seed_text

In [77]:
seed = "are you a "
generate_text = generate_text(lstm_model,tokenizer,seed,max_len,10)
print(generate_text)

are you a  very naughty you care about you you are the best


In [78]:
import pickle
with open("tokenizer.pkl", "wb") as f:
  pickle.dump(tokenizer, f)

In [79]:
with open("max_len.pkl", "wb") as f:
  pickle.dump(max_len, f)